# Papers Topic Hierarchy (Low → Mid → High)

Builds a three-level hierarchy for the existing **papers** topic model
by cutting BERTopic's agglomerative merge tree at two levels:

- **mid** — auto-selected by silhouette score over [HIGH_K_MAX + 1, n // 3]
- **high** — auto-selected by silhouette score over [HIGH_K_MIN, HIGH_K_MAX]

Inputs:
- `assets/topic_models/papers_topic_model`
- `assets/topic_models/papers_doc_topics.txt`
- `assets/topic_models/papers_topic_names.txt`
- `assets/embeddings/papers_corpus.txt`
- `assets/synbio_openalex.txt`

Outputs:
- `assets/reports/papers_topic_hierarchy_map.tsv` — document-level mapping (ID, low, mid, high)
- `assets/reports/papers_topic_name_hierarchy.tsv` — topic names mapped to hierarchy (global_name, low, mid, high)
- `assets/reports/papers_topic_hierarchy_summary.tsv` — mid/high group summary stats

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from bertopic import BERTopic

from hierarchy_utils import (
    build_hierarchy_maps,
    get_topic_embeddings,
    select_best_k,
    auto_mid_k_range,
    build_topic_hierarchy_df,
    build_doc_map,
    build_name_map,
    build_summary,
)

SEED = 42
np.random.seed(SEED)

ROOT_DIR = Path.cwd().parent
ASSETS_DIR = ROOT_DIR / "assets"
MODELS_DIR = ASSETS_DIR / "topic_models"
EMBEDDINGS_DIR = ASSETS_DIR / "embeddings"
REPORTS_DIR = ASSETS_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "papers_topic_model"
DOC_TOPICS_PATH = MODELS_DIR / "papers_doc_topics.txt"
TOPIC_NAMES_PATH = MODELS_DIR / "papers_topic_names.txt"
PAPERS_CORPUS_PATH = EMBEDDINGS_DIR / "papers_corpus.txt"
OPENALEX_PATH = ASSETS_DIR / "synbio_openalex.txt"

OUT_DOC_MAP = REPORTS_DIR / "papers_topic_hierarchy_map.tsv"
OUT_NAME_MAP = REPORTS_DIR / "papers_topic_name_hierarchy.tsv"
OUT_SUMMARY = REPORTS_DIR / "papers_topic_hierarchy_summary.tsv"

HIGH_K_MIN = 4
HIGH_K_MAX = 11

In [18]:
# Load model and input datasets
print("Loading model and datasets …")
papers_topic_model = BERTopic.load(str(MODEL_PATH))

papers_doc_topics = pd.read_csv(DOC_TOPICS_PATH, sep="\t")
papers_topic_names = pd.read_csv(TOPIC_NAMES_PATH, sep="\t")
papers_corpus = pd.read_csv(PAPERS_CORPUS_PATH, sep="\t")
papers_openalex = pd.read_csv(OPENALEX_PATH, sep="\t")

# Normalize key column names
papers_doc_topics = papers_doc_topics.rename(columns={"id": "ID", "topic": "low"})
papers_corpus = papers_corpus.rename(columns={"id": "ID"})
papers_openalex = papers_openalex.rename(columns={"id": "ID"})
papers_topic_names = papers_topic_names.rename(columns={"topic": "low"})

assert {"ID", "low"}.issubset(papers_doc_topics.columns)
assert {"low", "global_name"}.issubset(papers_topic_names.columns)
assert {"ID", "text"}.issubset(papers_corpus.columns)
assert {"ID", "publication_year"}.issubset(papers_openalex.columns)

papers_doc_topics["low"] = papers_doc_topics["low"].astype(int)
papers_topic_names["low"] = papers_topic_names["low"].astype(int)
papers_openalex["publication_year"] = pd.to_numeric(papers_openalex["publication_year"], errors="coerce")

print(f"  papers_doc_topics : {len(papers_doc_topics):,} rows")
print(f"  papers_topic_names: {len(papers_topic_names):,} rows")
print(f"  papers_corpus     : {len(papers_corpus):,} rows")
print(f"  papers_openalex   : {len(papers_openalex):,} rows")
print(f"  Outlier docs (low = -1): {(papers_doc_topics['low'] == -1).sum():,}")

Loading model and datasets …
  papers_doc_topics : 24,202 rows
  papers_topic_names: 263 rows
  papers_corpus     : 24,202 rows
  papers_openalex   : 24,202 rows
  Outlier docs (low = -1): 9,017


In [ ]:
# Build hierarchy and select levels
print("Building hierarchy …")
corpus_texts = papers_corpus["text"].astype(str).tolist()

hier_df, maps_by_k, low_topics = build_hierarchy_maps(
    papers_topic_model, corpus_texts,
)

# Compute topic embeddings once — reused for both level selections
embed_topics, X_emb = get_topic_embeddings(papers_topic_model, low_topics)

# Auto-select high-level k
selected_high_k, selected_high_score, high_scores = select_best_k(
    embed_topics, X_emb, maps_by_k,
    k_min=HIGH_K_MIN, k_max=HIGH_K_MAX, label="high",
)

# Auto-select mid-level k
MID_K_MIN, MID_K_MAX = auto_mid_k_range(len(low_topics), HIGH_K_MAX)
selected_mid_k, selected_mid_score, mid_scores = select_best_k(
    embed_topics, X_emb, maps_by_k,
    k_min=MID_K_MIN, k_max=MID_K_MAX, label="mid",
)

low_to_mid = maps_by_k[selected_mid_k]
low_to_high = maps_by_k[selected_high_k]

topic_hierarchy_map = build_topic_hierarchy_df(low_topics, low_to_mid, low_to_high)

# Display silhouette scores for both levels
print("\n--- High-level candidates ---")
display(pd.DataFrame(high_scores, columns=["high_k", "silhouette"]).sort_values("high_k"))
print("--- Mid-level candidates ---")
pd.DataFrame(mid_scores, columns=["mid_k", "silhouette"]).sort_values("mid_k")

Building hierarchy …
  Building BERTopic hierarchical merge tree …


100%|██████████| 262/262 [00:10<00:00, 24.78it/s]

  Non-outlier low topics: 263
  Tracking cluster maps for k = 1 … 40
  Captured 40 distinct k-level snapshots
  Scoring high-level k candidates in [4, 11] …
    k=  4  silhouette=0.1369
    k=  5  silhouette=0.1197
    k=  6  silhouette=0.1038
    k=  7  silhouette=0.0834
    k=  8  silhouette=0.0898
    k=  9  silhouette=0.1013
    k= 10  silhouette=0.0721
    k= 11  silhouette=0.0532
  ✓ Selected high-level k = 4 (silhouette = 0.1369)
  Hierarchy table: 263 low → 40 mid → 4 high


,high_k,silhouette
0,4,0.136893
1,5,0.119682
2,6,0.103818
3,7,0.083422
4,8,0.089843
5,9,0.101256
6,10,0.072098
7,11,0.053244


In [20]:
# Report 1: paper-level mapping (ID, low, mid, high)
print("Building document map …")
report1 = build_doc_map(papers_doc_topics, topic_hierarchy_map, id_col="ID")
report1.to_csv(OUT_DOC_MAP, sep="\t", index=False)

print(f"  Saved → {OUT_DOC_MAP.name}")
report1.head()

Building document map …
  Document map: 24,202 rows (9,017 outliers)
  Saved → papers_topic_hierarchy_map.tsv


,ID,low,mid,high
0,https://openalex.org/W2072812562,167,27,1
1,https://openalex.org/W2091309425,167,27,1
2,https://openalex.org/W2037457710,44,20,1
3,https://openalex.org/W1984248485,7,6,0
4,https://openalex.org/W1980762198,65,22,1


In [21]:
# Report 2: low-level global names mapped to low/mid/high
print("Building name map …")
report2 = build_name_map(papers_topic_names, topic_hierarchy_map)
report2.to_csv(OUT_NAME_MAP, sep="\t", index=False)

print(f"  Saved → {OUT_NAME_MAP.name}")
report2.head()

Building name map …
  Name map: 263 topics
  Saved → papers_topic_name_hierarchy.tsv


,global_name,low,mid,high
0,Genetic Circuit Control,0,0,0
1,Synthetic Nucleic Acid Design,1,1,1
2,Plant Biosynthetic Pathways,2,2,0
3,Synthetic CpG ODNs in Immunity,3,3,1
4,Ethics and Governance in Synthetic Biology,4,4,2


In [22]:
# Report 3: mid/high summaries with count, average year, median year
print("Building summary …")
report3 = build_summary(
    report1, papers_openalex,
    id_col="ID", year_col="publication_year",
)
report3.to_csv(OUT_SUMMARY, sep="\t", index=False)

print(f"  Saved → {OUT_SUMMARY.name}")
report3.head(10)

Building summary …
  Summary: 44 rows (mid + high)
  Saved → papers_topic_hierarchy_summary.tsv


,level,group_id,total_count,avg_publication_year,median_publication_year
0,high,0,7186,2017.192736,2019.0
1,high,1,6661,2004.748837,2007.0
2,high,2,1210,2016.650413,2017.0
3,high,3,128,2007.304688,2013.0
4,mid,0,1172,2015.867747,2016.0
5,mid,1,919,1996.373232,1995.0
6,mid,2,1055,2019.453081,2021.0
7,mid,3,338,2007.434911,2007.0
8,mid,4,1210,2016.650413,2017.0
9,mid,5,675,2008.022222,2010.0


In [ ]:
# Validation
assert list(report1.columns) == ["ID", "low", "mid", "high"]
assert list(report2.columns) == ["global_name", "low", "mid", "high"]
assert {"level", "group_id", "total_count", "avg_publication_year", "median_publication_year"}.issubset(report3.columns)
assert len(report1) == len(papers_doc_topics)
assert (report1.loc[report1["low"] == -1, ["mid", "high"]] == -1).all().all()
assert HIGH_K_MIN <= selected_high_k <= HIGH_K_MAX
assert MID_K_MIN <= selected_mid_k <= MID_K_MAX

print("All validation checks passed ✓")
print(f"  report1: {len(report1):,} rows | report2: {len(report2):,} rows | report3: {len(report3):,} rows")

All validation checks passed ✓
  report1: 24,202 rows | report2: 263 rows | report3: 44 rows


In [ ]:
# Hierarchy quality summary
outlier_pct = (papers_doc_topics["low"] == -1).mean() * 100

print(f"Selected high-level K : {selected_high_k}  (silhouette = {selected_high_score:.4f})")
print(f"Selected mid-level K  : {selected_mid_k}  (silhouette = {selected_mid_score:.4f})")
print(f"Outlier documents     : {outlier_pct:.2f}%")
print(f"Mid-level search range: [{MID_K_MIN}, {MID_K_MAX}]")

Selected high-level K : 4
Selected silhouette   : 0.1369
Outlier documents     : 37.26%


,high_k,silhouette
0,4,0.136893
1,5,0.119682
2,6,0.103818
3,7,0.083422
4,8,0.089843
5,9,0.101256
6,10,0.072098
7,11,0.053244
